# Faza I – Paraprocesimi i të dhënave (Eurostat)

**Dataset:** `estat_tour_occ_nim_en.csv` – netët e qëndrimit në akomodime turistike (Eurostat, `tour_occ_nim`), të dhëna mujore.
**Qëllimi:** analizimi i strukturës dhe cilësisë së të dhënave, pastrimi, trajtimi i vlerave që mungojnë, mostrimi, agregimi, integrimi, krijimi/përzgjedhja e veçorive, diskretizimi, binarizimi, transformimi dhe reduktimi i të dhënave.

## Përmbajtja
1. Mbledhja e të dhënave: burimi, API-ja, licenca, shprehjet e rregullta
2. Tipet e të dhënave: tipet e atributeve dhe lloji i datasetit
3. Cilësia e të dhënave: dimensionet e cilësisë, mungesat, dublikatat, vlerat ekstreme
4. Pastrimi, vlerat që mungojnë (me vlerësim të metodave) dhe vlerat ekstreme
5. Mostrimi (i rastësishëm dhe i shtresuar)
6. Agregimi vjetor
7. Integrimi: metadata e shteteve, bashkimi i njësive, redundanca (χ², korrelacioni, kovarianca)
8. Krijimi i veçorive
9. Transformimi: log, min-max, z-score, decimal scaling
10. Diskretizimi dhe binarizimi, smoothing me binning, hierarkitë e koncepteve
11. Zgjedhja e nënbashkësisë së veçorive (filtra + forward/backward + pema e vendimit)
12. Reduktimi i dimensionit (PCA) dhe kompresimi me wavelet (Haar)
13. Përmbledhja dhe eksporti final

## Lidhja me ligjëratat
| Java | Tema e ligjëratës | Seksionet |
|---|---|---|
| II | Grumbullimi i të dhënave: API, shprehjet e rregullta, licencat | 1.1 |
| III | Tipet e atributeve, cilësia (zhurma, outlier-ët, mungesat, dublikatat), agregimi, mostrimi, PCA, zgjedhja e veçorive | 2.1, 3, 4, 5, 6, 11, 12 |
| IV | Dimensionet e cilësisë, pastrimi si proces, integrimi, χ², korrelacioni, kovarianca | 3.1, 4.1, 7.1 |
| V | Reduktimi i të dhënave, normalizimi, diskretizimi, binning, hierarkitë e koncepteve, forward/backward selection, wavelet | 9, 10, 11.1, 12.1 |
| VI | Detektimi i anomalive – qasja statistikore | 4.2 |

## Si të përdoret
1. Vendoseni këtë notebook në të njëjtën dosje me skedarin CSV.
2. Instaloni bibliotekat nëse nevojiten: `pip install pandas numpy matplotlib scikit-learn scipy`.
3. Ekzekutoni qelizat nga lart-poshtë.
4. Rezultatet eksportohen në dosjen `output_phase1`.

## 1. Importimi i bibliotekave dhe ngarkimi i datasetit

In [ ]:
from pathlib import Path
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.feature_selection import VarianceThreshold, SequentialFeatureSelector
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score
from scipy.stats import chi2_contingency, chi2
from sklearn.decomposition import PCA

warnings.filterwarnings("ignore", category=FutureWarning)

INPUT_FILE = Path("estat_tour_occ_nim_en.csv")
if not INPUT_FILE.exists():
    fallback = Path("estat_tour_occ_nim_en.csv")
    if fallback.exists():
        INPUT_FILE = fallback
    else:
        raise FileNotFoundError("Vendoseni CSV-në në të njëjtën dosje me notebook-un ose ndryshoni INPUT_FILE.")

OUTPUT_DIR = Path("output_phase1")
OUTPUT_DIR.mkdir(exist_ok=True)

df_original = pd.read_csv(INPUT_FILE, low_memory=False)
df = df_original.copy()

print("Forma:", df.shape)
display(df.head())
print(df.columns.tolist())

### 1.1 Burimi i të dhënave, API-ja dhe licenca

Para mbledhjes së të dhënave i bëjmë tri pyetjet nga ligjërata:

| Pyetja | Përgjigjja për këtë dataset |
|---|---|
| **A janë të qasshme?** | Po – Eurostat i publikon falas, në ueb dhe përmes API-së REST (SDMX 2.1), pa çelës API. |
| **A janë të besueshme?** | Po – burim zyrtar statistikor i BE-së. Vlerat e vlerësuara ose me besueshmëri të ulët shënohen me `OBS_FLAG` (`e`, `u`, `b`). |
| **A është ligjore dhe etike përdorimi?** | Po – Eurostat lejon ripërdorimin me përmendjen e burimit (licenca CC BY 4.0). Të dhënat janë të agreguara, pa të dhëna personale (GDPR / Ligji 06/L-082 nuk preken). |

**Mënyra e mbledhjes:** API (jo crawling), sepse burimi ofron API zyrtare – ligjërata rekomandon API-të zyrtare kur ekzistojnë. API-ja kthen formatin SDMX-CSV, i njëjti format si skedari ynë.

In [ ]:
# API-ja e Eurostat (SDMX 2.1, REST + CSV). Filtri në URL: frekuenca.rezidenca.njësia.aktiviteti.shteti
API_BAZA = "https://ec.europa.eu/eurostat/api/dissemination/sdmx/2.1/data/tour_occ_nim"
API_URL = f"{API_BAZA}/M.TOTAL.NR.I551-I553.AL?format=SDMX-CSV&startPeriod=2024-01&endPeriod=2024-12"

SHKARKO_NGA_API = False  # vendoseni True për ta provuar shkarkimin (kërkon internet)

if SHKARKO_NGA_API:
    import urllib.request
    import io
    # Identifikojmë klientin me User-Agent, siç rekomandohet për mbledhje të përgjegjshme
    kerkesa = urllib.request.Request(API_URL, headers={"User-Agent": "PVDh-Grupi1 (projekt studentor)"})
    with urllib.request.urlopen(kerkesa, timeout=60) as pergjigja:
        df_api = pd.read_csv(io.BytesIO(pergjigja.read()))
    print("Rreshta nga API-ja:", len(df_api))
    print("Kolonat përputhen me CSV-në:", list(df_api.columns) == list(df_original.columns))
    display(df_api.head())
else:
    print("Shkarkimi nga API-ja është i çaktivizuar; përdoret skedari lokal:", INPUT_FILE)

**Shprehjet e rregullta (regex):** i përdorim për të nxjerrë informata nga tekstet dhe për të kontrolluar nëse vlerat i përmbahen formatit të pritur.

In [ ]:
# Nxjerrja e agjencisë, datasetit dhe versionit nga kolona DATAFLOW, p.sh. "ESTAT:TOUR_OCC_NIM(1.0)"
modeli_dataflow = r"^(?P<agjencia>\w+):(?P<dataseti>\w+)\((?P<versioni>[\d.]+)\)$"
display(df["DATAFLOW"].drop_duplicates().str.extract(modeli_dataflow))

# Kontrolli i formatit të kolonave me regex
kontrollet_regex = pd.DataFrame({
    "kolona": ["TIME_PERIOD", "geo", "nace_r2", "LAST UPDATE"],
    "modeli": [
        r"^\d{4}-(0[1-9]|1[0-2])$",            # VVVV-MM
        r"^([A-Z]{2}|EU\d{2}(_\d{4})?)$",      # kod shteti ose agregat EU
        r"^I55\d([-_]I55\d)?$",                # kodet NACE të akomodimit
        r"^\d{2}/\d{2}/\d{2} \d{2}:\d{2}:\d{2}$",  # dd/mm/vv hh:mm:ss
    ],
})
kontrollet_regex["vlera_jo_valide"] = [
    int((~df[k].astype(str).str.match(m)).sum())
    for k, m in zip(kontrollet_regex["kolona"], kontrollet_regex["modeli"])
]
display(kontrollet_regex)

## 2. Tipet e të dhënave dhe fjalori i variablave

In [ ]:
df.info()

type_report = pd.DataFrame({
    "column": df.columns,
    "dtype": [str(df[c].dtype) for c in df.columns],
    "unique_values": [df[c].nunique(dropna=True) for c in df.columns],
    "missing_values": [int(df[c].isna().sum()) for c in df.columns],
    "missing_percent": [(df[c].isna().mean() * 100).round(2) for c in df.columns],
})
display(type_report)
type_report.to_csv(OUTPUT_DIR / "01_data_types_report.csv", index=False)

column_descriptions = {
    "DATAFLOW": "Identifikuesi i rrjedhës së të dhënave",
    "LAST UPDATE": "Koha e përditësimit të datasetit",
    "freq": "Frekuenca (M = mujore)",
    "c_resid": "Kategoria e rezidentëve",
    "unit": "Njësia/treguesi statistikor",
    "nace_r2": "Kategoria e aktivitetit ekonomik",
    "geo": "Shteti/territori",
    "TIME_PERIOD": "Periudha kohore (vit-muaj)",
    "OBS_VALUE": "Vlera numerike e vëzhgimit",
    "OBS_FLAG": "Shenja/flag-u i vëzhgimit",
    "CONF_STATUS": "Statusi i konfidencialitetit",
}
variable_report = pd.DataFrame({
    "column": df.columns,
    "description": [column_descriptions.get(c, "Përshkrim për t'u verifikuar") for c in df.columns],
    "actual_dtype": [str(df[c].dtype) for c in df.columns],
    "unique_values": [df[c].nunique(dropna=True) for c in df.columns],
})
display(variable_report)
variable_report.to_csv(OUTPUT_DIR / "02_variable_dictionary.csv", index=False)

### 2.1 Tipet e atributeve dhe lloji i datasetit

Sipas ligjëratës, atributet ndahen në **kategorike** (nominale, ordinale) dhe **numerike** (interval, raport), si dhe në **diskrete** dhe **të vazhdueshme**.

In [ ]:
tipet_atributeve = pd.DataFrame([
    ("DATAFLOW", "Kategorik", "Nominal", "Diskret", "Identifikues i datasetit – vetëm etiketë (=, ≠)"),
    ("LAST UPDATE", "Numerik", "Interval", "Vazhdueshëm", "Data/ora: diferenca ka kuptim, zero e vërtetë nuk ka"),
    ("freq", "Kategorik", "Nominal", "Diskret", "Kodi i frekuencës (M)"),
    ("c_resid", "Kategorik", "Nominal", "Diskret", "DOM / FOR / TOTAL – pa renditje"),
    ("unit", "Kategorik", "Nominal", "Diskret", "Lloji i treguesit (NR, PCH_SM, ...)"),
    ("nace_r2", "Kategorik", "Nominal (hierarkik)", "Diskret", "I551-I553 përfshin I551 dhe I552_I553"),
    ("geo", "Kategorik", "Nominal (hierarkik)", "Diskret", "Shtetet përfshihen në agregatet EU"),
    ("TIME_PERIOD", "Numerik", "Interval", "Diskret", "Muajt: 2024-07 − 2023-07 = 12 muaj; s'ka zero të vërtetë"),
    ("OBS_VALUE (unit = NR)", "Numerik", "Raport", "Diskret", "Numër netësh: 0 = asnjë natë; 2 000 = 2 × 1 000"),
    ("OBS_VALUE (unit = PCH_SM*)", "Numerik", "Interval", "Vazhdueshëm", "Ndryshim në %: ka vlera negative, 0 = pa ndryshim"),
    ("OBS_FLAG", "Kategorik", "Nominal", "Diskret", "b, e, u, be – shënime cilësie"),
    ("CONF_STATUS", "Kategorik", "Nominal (binar)", "Diskret", "C = konfidenciale, ose bosh"),
], columns=["atributi", "kategoria", "lloji", "diskret / vazhdueshëm", "arsyetimi"])
display(tipet_atributeve)
tipet_atributeve.to_csv(OUTPUT_DIR / "02b_attribute_types.csv", index=False)

**Lloji i datasetit:** në formatin origjinal është **record data**: çdo rresht ka të njëjtat atribute, pra formon një matricë të dhënash. Në thelb janë **të dhëna të renditura**, më saktë **spatio-temporale**: vlerat varen nga shteti (hapësira) dhe nga muaji (koha). Renditja kohore ka rëndësi, prandaj imputimi (4.1), ndryshimi vjetor (8) dhe PCA (12) e përdorin strukturën e serisë kohore.

## 3. Cilësia e të dhënave: vlera që mungojnë, dublikata dhe vlera ekstreme

In [ ]:
missing_report = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2),
})
display(missing_report)
missing_report.to_csv(OUTPUT_DIR / "03_missing_values_report.csv")

full_duplicates = int(df.duplicated().sum())
key_columns = ["freq", "c_resid", "unit", "nace_r2", "geo", "TIME_PERIOD"]
key_duplicate_count = int(df.duplicated(subset=key_columns).sum())
print("Dublikatat e plota:", full_duplicates)
print("Dublikatat sipas dimensioneve kryesore:", key_duplicate_count)

for col in ["freq", "c_resid", "unit", "nace_r2", "geo", "OBS_FLAG", "CONF_STATUS"]:
    if col in df.columns:
        print(f"\nVlerat në {col}:")
        display(df[col].value_counts(dropna=False).head(15))

display(df["OBS_VALUE"].describe())

flag_missing_report = pd.crosstab(
    df["OBS_FLAG"].fillna("(flag mungon)"),
    df["OBS_VALUE"].isna(),
    rownames=["OBS_FLAG"],
    colnames=["OBS_VALUE mungon"],
    margins=True,
)
display(flag_missing_report)
flag_missing_report.to_csv(OUTPUT_DIR / "05_missing_obs_by_flag.csv")

obs_non_missing = df["OBS_VALUE"].dropna()
q1, q3 = obs_non_missing.quantile([0.25, 0.75])
iqr = q3 - q1
lower_bound, upper_bound = q1 - 1.5 * iqr, q3 + 1.5 * iqr
iqr_outliers = df[df["OBS_VALUE"].notna() & ((df["OBS_VALUE"] < lower_bound) | (df["OBS_VALUE"] > upper_bound))]
print("Vlera jashtë kufijve IQR (vetëm sinjal diagnostikues):", len(iqr_outliers))
iqr_outliers.head(1000).to_csv(OUTPUT_DIR / "06_potential_outliers_sample.csv", index=False)

### 3.1 Dimensionet e cilësisë së të dhënave

Ligjërata e javës IV e shikon cilësinë në gjashtë dimensione: saktësia, plotësia, konsistenca, aktualiteti, besueshmëria dhe interpretueshmëria. Për çdo dimension bëjmë kontroll konkret mbi datasetin. Kontrollet e saktësisë përdorin **metadata** (domenin dhe kufijtë e vlerave), siç kërkon hapi i parë i pastrimit si proces ("discrepancy detection").

In [ ]:
nr_vlerat = df.loc[df["unit"] == "NR", "OBS_VALUE"]
pch_vlerat = df.loc[df["unit"].str.startswith("PCH"), "OBS_VALUE"]

# Konsistenca: totalet duhet të jenë shuma e pjesëve
nr_df_kontroll = df[df["unit"] == "NR"]
p_resid = nr_df_kontroll.pivot_table(index=["geo", "nace_r2", "TIME_PERIOD"], columns="c_resid",
                                     values="OBS_VALUE", aggfunc="first").dropna()
kons_resid = ((p_resid["TOTAL"] - (p_resid["DOM"] + p_resid["FOR"])).abs() <= 1).mean() * 100

p_nace = nr_df_kontroll.pivot_table(index=["geo", "c_resid", "TIME_PERIOD"], columns="nace_r2",
                                    values="OBS_VALUE", aggfunc="first")
a = p_nace[["I551-I553", "I551", "I552_I553"]].dropna()
kons_nace = ((a["I551-I553"] - (a["I551"] + a["I552_I553"])).abs() <= 1).mean() * 100
b = p_nace[["I552_I553", "I552", "I553"]].dropna()
kons_nace2 = ((b["I552_I553"] - (b["I552"] + b["I553"])).abs() <= 1).mean() * 100

# Aktualiteti: data e përditësimit dhe periudha e fundit
data_perditesimit = pd.to_datetime(df["LAST UPDATE"].iloc[0], format="%d/%m/%y %H:%M:%S")
periudha_fundit = df["TIME_PERIOD"].max()

perqindja_flag = df["OBS_FLAG"].notna().mean() * 100

dimensionet_cilesise = pd.DataFrame([
    ("Saktësia", "NR < 0 (numër netësh negativ)", int((nr_vlerat < 0).sum())),
    ("Saktësia", "PCH_SM* < −100% (rënie më e madhe se 100%)", int((pch_vlerat < -100).sum())),
    ("Saktësia", "PCH_SM* > 1000% (rritje e dyshimtë, bazë e vogël)", int((pch_vlerat > 1000).sum())),
    ("Plotësia", "OBS_VALUE që mungojnë (%)", round(df["OBS_VALUE"].isna().mean() * 100, 2)),
    ("Konsistenca", "TOTAL = DOM + FOR (% e rreshtave)", round(kons_resid, 2)),
    ("Konsistenca", "I551-I553 = I551 + I552_I553 (%)", round(kons_nace, 2)),
    ("Konsistenca", "I552_I553 = I552 + I553 (%)", round(kons_nace2, 2)),
    ("Aktualiteti", "Data e përditësimit (LAST UPDATE)", data_perditesimit.strftime("%Y-%m-%d")),
    ("Aktualiteti", "Periudha e fundit në dataset", periudha_fundit),
    ("Besueshmëria", "Burimi", "Eurostat (zyrtar)"),
    ("Besueshmëria", "Rreshta me flag (e/u/b) (%)", round(perqindja_flag, 2)),
    ("Interpretueshmëria", "Kolona me kode që kërkojnë fjalor", "c_resid, unit, nace_r2, geo, OBS_FLAG"),
], columns=["dimensioni", "kontrolli", "rezultati"])
display(dimensionet_cilesise)
dimensionet_cilesise.to_csv(OUTPUT_DIR / "04_quality_dimensions.csv", index=False)

**Përfundime:**
- Nuk ka vlera të pamundura (NR negativ ose rënie nën −100%). Disa qindra rritje mbi 1000% vijnë nga një bazë shumë e vogël, p.sh. pas mbylljeve gjatë pandemisë. Ato janë **outlier-ë legjitimë**, jo zhurmë.
- Totalet janë konsistente me shumën e pjesëve në mbi 99% të rasteve. Kjo do të thotë që `TOTAL` dhe `I551-I553` janë **atribute të derivueshme** (redundancë, shih 7.1).
- **Zhurma vs outlier-ët:** zhurma është gabim i rastësishëm dhe zvogëlohet me smoothing (10.1). Outlier-ët mund të jenë vlera të vërteta, prandaj nuk i fshijmë verbërisht (4.2).

## 4. Pastrimi i të dhënave dhe strategjia për vlerat që mungojnë

In [ ]:
df_clean = df_original.copy()

for col in df_clean.select_dtypes(include=["object"]).columns:
    df_clean[col] = df_clean[col].map(lambda x: x.strip() if isinstance(x, str) else x)

df_clean["OBS_VALUE"] = pd.to_numeric(df_clean["OBS_VALUE"], errors="coerce")
df_clean["period_date"] = pd.to_datetime(
    df_clean["TIME_PERIOD"].astype(str) + "-01",
    format="%Y-%m-%d",
    errors="coerce",
)

df_analysis = df_clean[df_clean["OBS_VALUE"].notna()].copy()

constant_columns = [c for c in df_analysis.columns if df_analysis[c].nunique(dropna=False) <= 1]
df_analysis_reduced = df_analysis.drop(columns=constant_columns, errors="ignore")

print("Forma fillestare:", df_original.shape)
print("Forma pas pastrimit:", df_clean.shape)
print("Vlera OBS_VALUE që mungojnë:", df_clean["OBS_VALUE"].isna().sum())
print("Rreshta me OBS_VALUE të disponueshme:", len(df_analysis))
print("Kolona konstante në kopjen analitike:", constant_columns)

df_clean.to_csv(OUTPUT_DIR / "cleaned_dataset_with_missing_values.csv", index=False)
df_analysis.to_csv(OUTPUT_DIR / "analysis_dataset_non_missing.csv", index=False)

### 4.1 Strategjia për vlerat që mungojnë

Mungesat në `OBS_VALUE` kanë burime të ndryshme:
- **mungesa konfidenciale** – vlera është fshehur nga Eurostat (`CONF_STATUS = C`);
- **mungesa strukturore** – vlera nuk është raportuar (p.sh. shteti nuk ka mbledhur të dhëna për atë muaj ose për atë kategori);
- **mungesa e përqindjeve** – njësitë `PCH_SM*` nuk mund të llogariten kur mungon vlera e vitit krahasues.

Strategjia që zbatojmë:
1. Për analizën e përgjithshme rreshtat pa `OBS_VALUE` largohen (`df_analysis`), sepse vlera numerike është objekti i analizës.
2. Për serinë kohore kryesore (shtet × muaj, `NR`, `TOTAL`, `I551-I553`) boshllëqet e shkurtra (deri në 2 muaj) plotësohen me **interpolim linear** brenda serisë së secilit shtet.
3. Boshllëqet më të gjata plotësohen me **mesataren e të njëjtit muaj kalendarik** për atë shtet (imputim sezonal), sepse turizmi ka sezonalitet të fortë.
4. Seritë me më shumë se 30% mungesa në periudhën e analizës largohen, sepse imputimi do të ishte jo i besueshëm.
5. Kolonat kategorike: `OBS_FLAG` mungon → `"none"`, `CONF_STATUS` mungon → `"public"`.

In [ ]:
# Lidhja e mungesave me flag-un dhe statusin e konfidencialitetit
mungesat_sipas_statusit = (
    df_clean.assign(
        OBS_FLAG_i=df_clean["OBS_FLAG"].fillna("(mungon)"),
        CONF_STATUS_i=df_clean["CONF_STATUS"].fillna("(mungon)"),
        obs_mungon=df_clean["OBS_VALUE"].isna(),
    )
    .groupby(["CONF_STATUS_i", "OBS_FLAG_i"])["obs_mungon"]
    .agg(rreshta="count", obs_value_mungon="sum")
    .reset_index()
)
mungesat_sipas_statusit["perqindja_mungon"] = (
    mungesat_sipas_statusit["obs_value_mungon"] / mungesat_sipas_statusit["rreshta"] * 100
).round(2)
display(mungesat_sipas_statusit)

# Mungesat sipas njësisë
mungesat_sipas_njesise = df_clean.groupby("unit")["OBS_VALUE"].apply(lambda s: round(s.isna().mean() * 100, 2))
print("Përqindja e mungesave sipas njësisë:")
display(mungesat_sipas_njesise)

# Plotësimi i kolonave kategorike
df_clean["OBS_FLAG"] = df_clean["OBS_FLAG"].fillna("none")
df_clean["CONF_STATUS"] = df_clean["CONF_STATUS"].fillna("public")
df_analysis["OBS_FLAG"] = df_analysis["OBS_FLAG"].fillna("none")
df_analysis["CONF_STATUS"] = df_analysis["CONF_STATUS"].fillna("public")
mungesat_sipas_statusit.to_csv(OUTPUT_DIR / "07_missing_by_conf_status.csv", index=False)

In [ ]:
# Seria kryesore: netët totale (NR) në të gjitha akomodimet (I551-I553), të gjithë rezidentët (TOTAL)
AGREGATET_EU = ["EU25", "EU27_2007", "EU27_2020", "EU28"]
seria = df_clean[
    (df_clean["unit"] == "NR")
    & (df_clean["c_resid"] == "TOTAL")
    & (df_clean["nace_r2"] == "I551-I553")
    & (~df_clean["geo"].isin(AGREGATET_EU))
]
matrica = seria.pivot_table(index="geo", columns="period_date", values="OBS_VALUE", aggfunc="first", dropna=False)

# Periudha e analizës: 8 vitet e fundit të plota
viti_fundit = int(df_clean.loc[df_clean["period_date"].dt.month == 12, "period_date"].dt.year.max())
viti_fillim = viti_fundit - 7
muajt = pd.date_range(f"{viti_fillim}-01-01", f"{viti_fundit}-12-01", freq="MS")
matrica = matrica.reindex(columns=muajt)
print(f"Periudha e analizës: {viti_fillim}-01 deri {viti_fundit}-12 ({len(muajt)} muaj)")

perqindja_mungon = matrica.isna().mean(axis=1) * 100
display(perqindja_mungon.sort_values(ascending=False).round(1).to_frame("mungesa_%").head(15))

PRAGU_MUNGESES = 30
shtetet_larguara = perqindja_mungon[perqindja_mungon > PRAGU_MUNGESES].index.tolist()
matrica_para = matrica.drop(index=shtetet_larguara)
print("Shtetet e larguara (>30% mungesa):", shtetet_larguara)

# Hapi 1: interpolim linear për boshllëqe të shkurtra (deri në 2 muaj), vetëm brenda serisë
matrica_interp = matrica_para.T.interpolate(method="linear", limit=2, limit_area="inside").T

# Hapi 2: imputim sezonal – mesatarja e të njëjtit muaj kalendarik për atë shtet
matrica_imputuar = matrica_interp.copy()
for muaji in range(1, 13):
    kolonat = [c for c in matrica_imputuar.columns if c.month == muaji]
    mesatarja_muajit = matrica_imputuar[kolonat].mean(axis=1)
    for c in kolonat:
        matrica_imputuar[c] = matrica_imputuar[c].fillna(mesatarja_muajit)

# Kontroll: vlerat e imputuara
u_imputua = matrica_para.isna() & matrica_imputuar.notna()
print("Vlera që mungonin para imputimit:", int(matrica_para.isna().sum().sum()))
print("  - plotësuar me interpolim:", int((matrica_para.isna() & matrica_interp.notna()).sum().sum()))
print("  - plotësuar me mesataren sezonale:", int((matrica_interp.isna() & matrica_imputuar.notna()).sum().sum()))
print("Vlera që mungojnë pas imputimit:", int(matrica_imputuar.isna().sum().sum()))

# Formati i gjatë me kolonën was_imputed
seria_imputuar = (
    matrica_imputuar.stack().rename("OBS_VALUE").to_frame()
    .join(u_imputua.stack().rename("was_imputed"))
    .reset_index()
    .rename(columns={"level_1": "period_date"})
)
seria_imputuar.to_csv(OUTPUT_DIR / "08_main_series_imputed.csv", index=False)

# Krahasim para/pas për një shtet me mungesa
shteti_shembull = (matrica_para.isna().sum(axis=1)).idxmax()
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(matrica_imputuar.columns, matrica_imputuar.loc[shteti_shembull], "--", color="tab:orange", label="pas imputimit")
ax.plot(matrica_para.columns, matrica_para.loc[shteti_shembull], color="tab:blue", label="origjinale")
ax.set_title(f"Seria e netëve – {shteti_shembull} (para dhe pas imputimit)")
ax.set_ylabel("Netë qëndrimi")
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig_01_imputimi.png", dpi=120)
plt.show()

**Vlerësimi i metodave të imputimit.** Ligjërata liston disa mënyra plotësimi: konstante globale, mesatarja e atributit, *mesatarja e atributit për të njëjtën klasë* ("më e mençur") dhe vlera më e mundshme. I krahasojmë empirikisht: fshehim 5% të vlerave të njohura, i plotësojmë me secilën metodë dhe matim gabimin (MAPE). Për serinë tonë "klasa" është kombinimi **shtet × muaj kalendarik**.

In [ ]:
e_njohura = matrica_para.stack()
indeksi_test = e_njohura.sample(frac=0.05, random_state=42).index
vlerat_vertete = e_njohura.loc[indeksi_test]

matrica_test = matrica_para.copy()
for geo_i, data_i in indeksi_test:
    matrica_test.loc[geo_i, data_i] = np.nan


def plotso_sezonale(m):
    # mesatarja e të njëjtit muaj kalendarik për të njëjtin shtet
    m = m.copy()
    for muaji in range(1, 13):
        kol = [c for c in m.columns if c.month == muaji]
        mes = m[kol].mean(axis=1)
        for c in kol:
            m[c] = m[c].fillna(mes)
    return m


metodat_imputimit = {
    "Konstante globale (mesatarja e gjithë matricës)": matrica_test.fillna(matrica_test.stack().mean()),
    "Mesatarja e atributit (mesatarja e shtetit)": matrica_test.T.fillna(matrica_test.mean(axis=1)).T,
    "Mesatarja e klasës (shtet × muaj kalendarik)": plotso_sezonale(matrica_test),
    "Interpolim linear + mesatarja e klasës (metoda jonë)": plotso_sezonale(
        matrica_test.T.interpolate(method="linear", limit=2, limit_area="inside").T),
}

rezultatet_imputimit = []
for emri, m in metodat_imputimit.items():
    parashikimi = m.stack().reindex(indeksi_test)
    gabimi = ((parashikimi - vlerat_vertete).abs() / vlerat_vertete.replace(0, np.nan)).mean() * 100
    rezultatet_imputimit.append((emri, round(gabimi, 2)))
rezultatet_imputimit = pd.DataFrame(rezultatet_imputimit, columns=["metoda", "MAPE_%"]).sort_values("MAPE_%")
print("Vlera të fshehura për test:", len(indeksi_test))
display(rezultatet_imputimit)
rezultatet_imputimit.to_csv(OUTPUT_DIR / "08b_imputation_comparison.csv", index=False)

### 4.2 Trajtimi i vlerave ekstreme

Vlerat jashtë kufijve IQR (seksioni 3) **nuk janë gabime**: ato i përkasin shteteve të mëdha (DE, ES, FR, IT) dhe agregateve të BE-së, ose muajve të sezonit të lartë. Prandaj ato **nuk fshihen**. Ndikimi i tyre zvogëlohet:
- me transformimin logaritmik `log1p` (seksioni 9);
- me winsorizim në përqindjet 1%–99% si kolonë e veçantë (`OBS_VALUE_winsor`), pa e ndryshuar vlerën origjinale;
- me largimin e agregateve EU nga analiza në nivel shteti (seksioni 7).

Në fund përdorim edhe **qasjen statistikore** të detektimit të anomalive (java VI): një vlerë konsiderohet anomali nëse |z| > 3. Z-score llogaritet brenda serisë së secilit shtet, sepse e njëjta vlerë mund të jetë normale për Gjermaninë dhe anomale për Maltën (**anomali kontekstuale**).

In [ ]:
outliers_sipas_geo = iqr_outliers["geo"].value_counts().head(10)
print("Shtetet me më shumë vlera ekstreme (IQR):")
display(outliers_sipas_geo)

# Winsorizim brenda secilës njësi (NR dhe përqindjet kanë shkallë të ndryshme)
p_ulet = df_analysis.groupby("unit")["OBS_VALUE"].transform(lambda s: s.quantile(0.01))
p_larte = df_analysis.groupby("unit")["OBS_VALUE"].transform(lambda s: s.quantile(0.99))
df_analysis["OBS_VALUE_winsor"] = df_analysis["OBS_VALUE"].clip(lower=p_ulet, upper=p_larte)

ndryshuar = (df_analysis["OBS_VALUE_winsor"] != df_analysis["OBS_VALUE"]).sum()
print(f"Vlera të kufizuara nga winsorizimi: {ndryshuar} ({ndryshuar / len(df_analysis) * 100:.2f}%)")
display(df_analysis.groupby("unit")[["OBS_VALUE", "OBS_VALUE_winsor"]].agg(["min", "max"]))

In [ ]:
seria_log = np.log1p(matrica_imputuar)
z_shteti = seria_log.sub(seria_log.mean(axis=1), axis=0).div(seria_log.std(axis=1), axis=0)

anomalite = (
    z_shteti.stack().rename("z").reset_index()
    .rename(columns={"level_1": "period_date"})
    .loc[lambda d: d["z"].abs() > 3]
    .sort_values("z")
)
anomalite["NR"] = [matrica_imputuar.loc[g, p] for g, p in zip(anomalite["geo"], anomalite["period_date"])]
print("Anomali me |z| > 3:", len(anomalite))
print("Sipas vitit:")
display(anomalite["period_date"].dt.year.value_counts().sort_index().to_frame("anomali"))
display(anomalite.head(15).round(2))
anomalite.to_csv(OUTPUT_DIR / "06b_statistical_anomalies.csv", index=False)

Pothuajse të gjitha anomalitë bien në pranverën e vitit 2020 (mbyllja gjatë pandemisë COVID-19). Ky shkak njihet dhe nuk është gabim matjeje, prandaj vlerat **mbahen**: ato janë pikërisht informata që analiza duhet të kapë.

## 5. Mostrimi

In [ ]:
sample_size = min(5000, len(df_analysis))
df_sample = df_analysis.sample(n=sample_size, random_state=42)
df_sample.to_csv(OUTPUT_DIR / "random_sample.csv", index=False)
print("Madhësia e mostrës:", len(df_sample))
display(df_sample.head())

### 5.1 Mostrimi i shtresuar

Mostra e rastësishme mund të mos i ruajë proporcionet e kategorive. Me mostrimin e shtresuar marrim të njëjtën përqindje nga çdo kombinim `unit` × `c_resid`, kështu që shpërndarja e mostrës përputhet me atë të popullacionit.

In [ ]:
fraksioni = sample_size / len(df_analysis)
df_sample_shtresuar = (
    df_analysis.groupby(["unit", "c_resid"], group_keys=False)
    .sample(frac=fraksioni, random_state=42)
)
print("Madhësia e mostrës së shtresuar:", len(df_sample_shtresuar))

krahasimi_mostrave = pd.DataFrame({
    "popullacioni_%": df_analysis.groupby(["unit", "c_resid"]).size() / len(df_analysis) * 100,
    "mostra_rastesishme_%": df_sample.groupby(["unit", "c_resid"]).size() / len(df_sample) * 100,
    "mostra_shtresuar_%": df_sample_shtresuar.groupby(["unit", "c_resid"]).size() / len(df_sample_shtresuar) * 100,
}).round(2)
krahasimi_mostrave["devijimi_rastesishme"] = (krahasimi_mostrave["mostra_rastesishme_%"] - krahasimi_mostrave["popullacioni_%"]).abs().round(2)
krahasimi_mostrave["devijimi_shtresuar"] = (krahasimi_mostrave["mostra_shtresuar_%"] - krahasimi_mostrave["popullacioni_%"]).abs().round(2)
display(krahasimi_mostrave)
print("Devijimi mesatar – e rastësishme:", krahasimi_mostrave["devijimi_rastesishme"].mean().round(3),
      "| e shtresuar:", krahasimi_mostrave["devijimi_shtresuar"].mean().round(3))

df_sample_shtresuar.to_csv(OUTPUT_DIR / "stratified_sample.csv", index=False)
krahasimi_mostrave.to_csv(OUTPUT_DIR / "09_sampling_comparison.csv")

## 6. Agregimi vjetor

In [ ]:
df_for_annual = df_analysis.copy()
df_for_annual["year"] = df_for_annual["period_date"].dt.year
annual_keys = ["geo", "c_resid", "unit", "nace_r2"]

nr_df = df_for_annual[df_for_annual["unit"] == "NR"]
other_df = df_for_annual[df_for_annual["unit"] != "NR"]

annual_parts = []
if not nr_df.empty:
    annual_nr = nr_df.groupby(annual_keys + ["year"], dropna=False).agg(
        annual_value=("OBS_VALUE", "sum"),
        available_months=("OBS_VALUE", "count")
    ).reset_index()
    annual_nr["aggregation_method"] = "sum_monthly_NR"
    annual_parts.append(annual_nr)

if not other_df.empty:
    annual_other = other_df.groupby(annual_keys + ["year"], dropna=False).agg(
        annual_value=("OBS_VALUE", "mean"),
        available_months=("OBS_VALUE", "count")
    ).reset_index()
    annual_other["aggregation_method"] = "mean_monthly_non_NR"
    annual_parts.append(annual_other)

annual_summary = pd.concat(annual_parts, ignore_index=True) if annual_parts else pd.DataFrame()
if not annual_summary.empty:
    annual_summary["expected_months"] = np.where(
        annual_summary["year"] == df_for_annual["year"].max(),
        df_for_annual.groupby("year")["TIME_PERIOD"].nunique().get(df_for_annual["year"].max(), 12),
        12
    )
    annual_summary["months_coverage_percent"] = (
        annual_summary["available_months"] / annual_summary["expected_months"].replace(0, np.nan) * 100
    ).round(2)
    annual_summary.to_csv(OUTPUT_DIR / "annual_aggregated_summary.csv", index=False)
display(annual_summary.head())

**Agregimi e bën të dhënën më stabile.** Sipas ligjëratës, të dhënat e agreguara kanë më pak variabilitet. E kontrollojmë duke krahasuar koeficientin e variacionit (CV = σ / μ) të vlerave mujore dhe vjetore për çdo shtet. Agregimi është njëkohësisht **reduktim i numrit të objekteve**: 12 muaj → 1 vit.

In [ ]:
filtri_seria = (lambda d: d[(d["unit"] == "NR") & (d["c_resid"] == "TOTAL") & (d["nace_r2"] == "I551-I553")
                            & (~d["geo"].isin(AGREGATET_EU))])
mujore = filtri_seria(df_for_annual)
vjetore = filtri_seria(annual_summary)
vjetore = vjetore[vjetore["available_months"] == 12]

cv = lambda s: s.std() / s.mean()
krahasimi_cv = pd.DataFrame({
    "CV_mujor": mujore.groupby("geo")["OBS_VALUE"].apply(cv),
    "CV_vjetor": vjetore.groupby("geo")["annual_value"].apply(cv),
}).dropna().round(3)
display(krahasimi_cv.head(10))
print("CV mesatar – mujor:", krahasimi_cv["CV_mujor"].mean().round(3),
      "| vjetor:", krahasimi_cv["CV_vjetor"].mean().round(3))
print("Rreshta para agregimit:", len(df_for_annual), "| pas agregimit:", len(annual_summary))

## 7. Integrimi i të dhënave

Integrimi bëhet në dy mënyra:
1. **Integrim me burim të jashtëm** – bashkojmë datasetin me një tabelë metadata për shtetet (emri, rajoni, anëtarësia në BE, nëse kodi është agregat).
2. **Integrim i njësive** – vlerat `NR`, `PCH_SM`, `PCH_SM_19`, `PCH_SM_2Y` janë në rreshta të ndara; i bashkojmë në një rresht për çdo çelës `geo, c_resid, nace_r2, TIME_PERIOD`.

Pas integrimit agregatet EU largohen nga analiza në nivel shteti, që vlerat të mos numërohen dy herë.

**Problemi i identifikimit të entiteteve:** kodet e Eurostat-it ndryshojnë nga ISO 3166 për disa shtete. Greqia është `EL` (ISO: `GR`), Mbretëria e Bashkuar është `UK` (ISO: `GB`), dhe Kosova `XK` (kod i përkohshëm). Tabela metadata i lidh këto kode me emrat, që bashkimi me burime të tjera të bëhet saktë. Agregatet EU25, EU27_2007, EU27_2020 dhe EU28 janë entitete të ndryshme që mbivendosen me shtetet.

In [ ]:
# Tabela metadata për shtetet (burimi: kodet e Eurostat dhe anëtarësia në BE që nga 2020)
metadata_shtetet = pd.DataFrame([
    ("AL", "Shqipëria", "Ballkani Perëndimor", False), ("AT", "Austria", "Evropa Perëndimore", True),
    ("BE", "Belgjika", "Evropa Perëndimore", True), ("BG", "Bullgaria", "Evropa Lindore", True),
    ("CH", "Zvicra", "Evropa Perëndimore", False), ("CY", "Qiproja", "Evropa Jugore", True),
    ("CZ", "Çekia", "Evropa Lindore", True), ("DE", "Gjermania", "Evropa Perëndimore", True),
    ("DK", "Danimarka", "Evropa Veriore", True), ("EE", "Estonia", "Evropa Lindore", True),
    ("EL", "Greqia", "Evropa Jugore", True), ("ES", "Spanja", "Evropa Jugore", True),
    ("FI", "Finlanda", "Evropa Veriore", True), ("FR", "Franca", "Evropa Perëndimore", True),
    ("HR", "Kroacia", "Evropa Jugore", True), ("HU", "Hungaria", "Evropa Lindore", True),
    ("IE", "Irlanda", "Evropa Perëndimore", True), ("IS", "Islanda", "Evropa Veriore", False),
    ("IT", "Italia", "Evropa Jugore", True), ("LI", "Lihtenshtajni", "Evropa Perëndimore", False),
    ("LT", "Lituania", "Evropa Lindore", True), ("LU", "Luksemburgu", "Evropa Perëndimore", True),
    ("LV", "Letonia", "Evropa Lindore", True), ("MD", "Moldavia", "Evropa Lindore", False),
    ("ME", "Mali i Zi", "Ballkani Perëndimor", False), ("MK", "Maqedonia e Veriut", "Ballkani Perëndimor", False),
    ("MT", "Malta", "Evropa Jugore", True), ("NL", "Holanda", "Evropa Perëndimore", True),
    ("NO", "Norvegjia", "Evropa Veriore", False), ("PL", "Polonia", "Evropa Lindore", True),
    ("PT", "Portugalia", "Evropa Jugore", True), ("RO", "Rumania", "Evropa Lindore", True),
    ("RS", "Serbia", "Ballkani Perëndimor", False), ("SE", "Suedia", "Evropa Veriore", True),
    ("SI", "Sllovenia", "Evropa Jugore", True), ("SK", "Sllovakia", "Evropa Lindore", True),
    ("TR", "Turqia", "Tjetër", False), ("UK", "Mbretëria e Bashkuar", "Evropa Perëndimore", False),
    ("XK", "Kosova", "Ballkani Perëndimor", False),
    ("EU25", "BE (25 shtete)", "Agregat", False), ("EU27_2007", "BE (27 shtete, 2007-2013)", "Agregat", False),
    ("EU27_2020", "BE (27 shtete, nga 2020)", "Agregat", False), ("EU28", "BE (28 shtete, 2013-2020)", "Agregat", False),
], columns=["geo", "emri_shtetit", "rajoni", "anetare_e_BE"])
metadata_shtetet["eshte_agregat"] = metadata_shtetet["rajoni"] == "Agregat"

df_integruar = df_analysis.merge(metadata_shtetet, on="geo", how="left", validate="many_to_one", indicator=True)
pa_perputhje = df_integruar.loc[df_integruar["_merge"] != "both", "geo"].unique()
print("Kode geo pa përputhje në metadata:", list(pa_perputhje))
assert len(pa_perputhje) == 0, "Disa kode geo nuk kanë metadata"
df_integruar = df_integruar.drop(columns="_merge")
print("Forma pas integrimit me metadata:", df_integruar.shape)
display(df_integruar.groupby("rajoni")["geo"].nunique().to_frame("numri_i_shteteve"))

In [ ]:
# Integrimi i njësive: një rresht për çdo çelës, një kolonë për çdo njësi
celesi = ["geo", "c_resid", "nace_r2", "TIME_PERIOD", "period_date"]
df_njesite = (
    df_integruar.pivot_table(index=celesi, columns="unit", values="OBS_VALUE", aggfunc="first")
    .reset_index()
)
df_njesite.columns.name = None

# Flag-u: shënojmë nëse ndonjë nga vlerat e çelësit ka flag
flag_sipas_celesit = (
    df_integruar.assign(ka_flag=df_integruar["OBS_FLAG"] != "none")
    .groupby(celesi)["ka_flag"].any().reset_index()
)
df_njesite = df_njesite.merge(flag_sipas_celesit, on=celesi, how="left", validate="one_to_one")
df_njesite = df_njesite.merge(metadata_shtetet, on="geo", how="left", validate="many_to_one")

print("Rreshta para integrimit të njësive:", len(df_integruar))
print("Rreshta pas integrimit të njësive:", len(df_njesite))
print("Plotësia e kolonave të njësive (%):")
display((df_njesite[["NR", "PCH_SM", "PCH_SM_19", "PCH_SM_2Y"]].notna().mean() * 100).round(2))

# Largimi i agregateve EU nga analiza në nivel shteti
df_shtetet = df_njesite[~df_njesite["eshte_agregat"]].copy()
print("Rreshta në nivel shteti (pa agregate EU):", len(df_shtetet))
df_njesite.to_csv(OUTPUT_DIR / "10_integrated_dataset.csv", index=False)

### 7.1 Redundanca gjatë integrimit: χ², korrelacioni dhe kovarianca

- **Atributet nominale:** me testin **χ²** kontrollojmë nëse dy atribute kategorike janë të varura. Llogaritja bëhet manualisht si në ligjëratë (frekuenca e pritur = totali i rreshtit × totali i kolonës / totali i përgjithshëm) dhe pastaj verifikohet me `scipy`.
- **Atributet numerike:** me **korrelacionin e Pearson-it** dhe **kovariancën** zbulojmë atributet e derivueshme, si `TOTAL = DOM + FOR`.

In [ ]:
# χ² për 2×2: anëtarësia në BE × prania e flag-ut
tabela_obs = pd.crosstab(df_shtetet["anetare_e_BE"], df_shtetet["ka_flag"])
tabela_obs.index = ["Jo anëtare", "Anëtare e BE"]
tabela_obs.columns = ["Pa flag", "Me flag"]

totali = tabela_obs.values.sum()
tabela_pritur = np.outer(tabela_obs.sum(axis=1), tabela_obs.sum(axis=0)) / totali
chi2_manual = (((tabela_obs.values - tabela_pritur) ** 2) / tabela_pritur).sum()
shkallet_lirise = (tabela_obs.shape[0] - 1) * (tabela_obs.shape[1] - 1)
vlera_kritike = chi2.ppf(1 - 0.001, shkallet_lirise)

print("Frekuencat e vëzhguara:")
display(tabela_obs)
print("Frekuencat e pritura:")
display(pd.DataFrame(tabela_pritur, index=tabela_obs.index, columns=tabela_obs.columns).round(1))
chi2_scipy, p_vlera, _, _ = chi2_contingency(tabela_obs, correction=False)
print(f"χ² (manual) = {chi2_manual:.2f} | χ² (scipy) = {chi2_scipy:.2f} | shkallët e lirisë = {shkallet_lirise}")
print(f"Vlera kritike në nivelin 0.001: {vlera_kritike:.3f} | p = {p_vlera:.2e}")
print("Përfundimi:", "atributet janë të varura (hipoteza e pavarësisë refuzohet)" if chi2_manual > vlera_kritike
      else "nuk mund të tregohet varësi")

# χ² për tabelë më të madhe: rajoni × prania e flag-ut
tabela_rajoni = pd.crosstab(df_shtetet["rajoni"], df_shtetet["ka_flag"])
chi2_r, p_r, dof_r, _ = chi2_contingency(tabela_rajoni)
print(f"\nRajoni × flag: χ² = {chi2_r:.2f}, shkallët e lirisë = {dof_r}, p = {p_r:.2e}")

In [ ]:
# Korrelacioni dhe kovarianca: a është TOTAL atribut i derivueshëm nga DOM dhe FOR?
nr_rezidenca = (
    df_shtetet[df_shtetet["nace_r2"] == "I551-I553"]
    .pivot_table(index=["geo", "period_date"], columns="c_resid", values="NR", aggfunc="first")
    .dropna()
)
A, B = nr_rezidenca["DOM"], nr_rezidenca["FOR"]

# Kovarianca me formulën e ligjëratës: Cov(A,B) = E(A·B) − E(A)·E(B)
cov_manual = (A * B).mean() - A.mean() * B.mean()
r_manual = cov_manual / (A.std(ddof=0) * B.std(ddof=0))
print(f"Cov(DOM, FOR) = {cov_manual:.3e}  (> 0: rriten bashkë)")
print(f"r(DOM, FOR) = {r_manual:.3f}  | numpy: {np.corrcoef(A, B)[0, 1]:.3f}")
print(f"r(TOTAL, DOM + FOR) = {np.corrcoef(nr_rezidenca['TOTAL'], A + B)[0, 1]:.6f}")
print("→ TOTAL është atribut i derivueshëm (redundant) nga DOM dhe FOR. E mbajmë si rresht më vete,")
print("  sepse analizat e mëtejshme e përdorin drejtpërdrejt, por nuk e trajtojmë si informatë të re.")

display(nr_rezidenca[["DOM", "FOR", "TOTAL"]].corr().round(3))
display(nr_rezidenca[["DOM", "FOR", "TOTAL"]].cov().apply(lambda s: s.map(lambda v: f"{v:.2e}")))

## 8. Krijimi i veçorive

Nga data dhe nga njësitë krijojmë veçori të reja:
- `year`, `month`, `quarter`, `season` – komponentët kohorë;
- `is_peak_season` – muajt qershor–gusht;
- `foreign_share` – pjesa e netëve të turistëve të huaj ndaj totalit (`FOR / TOTAL`);
- `yoy_change_calc` – ndryshimi vjetor i llogaritur nga ne, që e krahasojmë me `PCH_SM` të Eurostat-it (kontroll i cilësisë);
- `rolling_3m_mean` – mesatarja lëvizëse 3-mujore e netëve.

In [ ]:
df_features = df_shtetet.copy()

# Veçoritë kohore
df_features["year"] = df_features["period_date"].dt.year
df_features["month"] = df_features["period_date"].dt.month
df_features["quarter"] = df_features["period_date"].dt.quarter
stinet = {12: "Dimër", 1: "Dimër", 2: "Dimër", 3: "Pranverë", 4: "Pranverë", 5: "Pranverë",
          6: "Verë", 7: "Verë", 8: "Verë", 9: "Vjeshtë", 10: "Vjeshtë", 11: "Vjeshtë"}
df_features["season"] = df_features["month"].map(stinet)
df_features["is_peak_season"] = df_features["month"].isin([6, 7, 8]).astype(int)

# Pjesa e turistëve të huaj: FOR / TOTAL për të njëjtin shtet, aktivitet dhe muaj
nr_resid = df_features.pivot_table(index=["geo", "nace_r2", "period_date"], columns="c_resid", values="NR", aggfunc="first")
nr_resid["foreign_share"] = nr_resid["FOR"] / nr_resid["TOTAL"].replace(0, np.nan)
df_features = df_features.merge(
    nr_resid[["foreign_share"]].reset_index(), on=["geo", "nace_r2", "period_date"], how="left", validate="many_to_one"
)

# Ndryshimi vjetor i llogaritur: krahasim me të njëjtin muaj të vitit të kaluar
grupi = ["geo", "c_resid", "nace_r2"]
vlera_vitit_kaluar = df_features[grupi + ["period_date", "NR"]].copy()
vlera_vitit_kaluar["period_date"] = vlera_vitit_kaluar["period_date"] + pd.DateOffset(years=1)
vlera_vitit_kaluar = vlera_vitit_kaluar.rename(columns={"NR": "NR_viti_kaluar"})
df_features = df_features.merge(vlera_vitit_kaluar, on=grupi + ["period_date"], how="left", validate="one_to_one")
df_features["yoy_change_calc"] = (
    (df_features["NR"] / df_features["NR_viti_kaluar"].replace(0, np.nan) - 1) * 100
).round(1)

# Mesatarja lëvizëse 3-mujore brenda secilës seri
df_features = df_features.sort_values(grupi + ["period_date"]).reset_index(drop=True)
df_features["rolling_3m_mean"] = (
    df_features.groupby(grupi)["NR"].transform(lambda s: s.rolling(3, min_periods=1).mean())
)

# Kontroll: ndryshimi ynë vs PCH_SM i Eurostat-it
krahasimi = df_features[["yoy_change_calc", "PCH_SM"]].dropna()
diferenca = (krahasimi["yoy_change_calc"] - krahasimi["PCH_SM"]).abs()
print("Rreshta të krahasueshëm:", len(krahasimi))
print("Korrelacioni yoy_change_calc vs PCH_SM:", round(krahasimi.corr().iloc[0, 1], 4))
print("Përputhje brenda ±0.5 pikë përqindjeje:", round((diferenca <= 0.5).mean() * 100, 2), "%")

display(df_features[["geo", "c_resid", "nace_r2", "period_date", "NR", "season", "is_peak_season",
                     "foreign_share", "yoy_change_calc", "PCH_SM", "rolling_3m_mean"]].dropna().head(10))

## 9. Transformimi

Netët e qëndrimit (`NR`) kanë shpërndarje shumë asimetrike (shtetet e mëdha kanë vlera qindra herë më të mëdha). Zbatojmë:
- **log1p** – zvogëlon asimetrinë dhe ndikimin e vlerave ekstreme;
- **standardizimin (z-score)** me `StandardScaler` – mesatarja 0, devijimi standard 1;
- **normalizimin min-max** me `MinMaxScaler` – vlerat në intervalin [0, 1]: v' = (v − min) / (max − min);
- **decimal scaling** – v' = v / 10^j, ku j është numri më i vogël i plotë për të cilin max(|v'|) < 1.

Normalizimi u jep të gjitha atributeve peshë të barabartë. Kjo është e rëndësishme për metodat që bazohen në distancë (k-NN, clustering) dhe për PCA.

In [ ]:
ka_nr = df_features["NR"].notna()
df_features["NR_log"] = np.log1p(df_features["NR"])
df_features.loc[ka_nr, "NR_log_zscore"] = StandardScaler().fit_transform(df_features.loc[ka_nr, ["NR_log"]]).ravel()
df_features.loc[ka_nr, "NR_minmax"] = MinMaxScaler().fit_transform(df_features.loc[ka_nr, ["NR"]]).ravel()

# Decimal scaling: j = numri më i vogël i plotë që max(|v'|) < 1
j_decimal = int(np.floor(np.log10(df_features["NR"].abs().max()))) + 1
df_features["NR_decimal"] = df_features["NR"] / 10 ** j_decimal
print("Decimal scaling: j =", j_decimal, "| max |v'| =", round(df_features["NR_decimal"].abs().max(), 4))

# Verifikim manual i formulave për një vlerë
v = df_features.loc[ka_nr, "NR"].iloc[0]
nr_min, nr_max = df_features["NR"].min(), df_features["NR"].max()
print(f"Shembull: v = {v:,.0f} → min-max = {(v - nr_min) / (nr_max - nr_min):.6f} "
      f"(sklearn: {df_features.loc[ka_nr, 'NR_minmax'].iloc[0]:.6f})")
mu, sigma = df_features["NR_log"].mean(), df_features["NR_log"].std(ddof=0)
print(f"           log1p(v) = {np.log1p(v):.4f} → z = {(np.log1p(v) - mu) / sigma:.4f} "
      f"(sklearn: {df_features.loc[ka_nr, 'NR_log_zscore'].iloc[0]:.4f})")


display(df_features[["NR", "NR_log", "NR_log_zscore", "NR_minmax", "NR_decimal"]].describe().round(3))
print("Asimetria (skewness) – NR:", round(df_features["NR"].skew(), 2), "| NR_log:", round(df_features["NR_log"].skew(), 2))

fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
axes[0].hist(df_features["NR"].dropna(), bins=60, color="tab:blue")
axes[0].set_title("NR (origjinale)")
axes[1].hist(df_features["NR_log"].dropna(), bins=60, color="tab:green")
axes[1].set_title("log1p(NR)")
axes[2].hist(df_features["NR_log_zscore"].dropna(), bins=60, color="tab:purple")
axes[2].set_title("z-score i log1p(NR)")
for ax in axes:
    ax.set_ylabel("Frekuenca")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig_02_transformimi.png", dpi=120)
plt.show()

## 10. Diskretizimi dhe binarizimi

**Diskretizimi** – vlera e vazhdueshme `NR` ndahet në 4 klasa:
- me **frekuencë të barabartë** (`pd.qcut`) – çdo klasë ka afërsisht të njëjtin numër rreshtash;
- me **gjerësi të barabartë** (`pd.cut`) – intervale me të njëjtën gjatësi; për të dhëna asimetrike pothuajse të gjitha vlerat bien në klasën e parë. Prandaj e zbatojmë edhe mbi `log1p(NR)`.

**Binarizimi** – krijojmë veçori 0/1: `growth_positive` (`PCH_SM > 0`), `is_peak_season`, `has_flag`, si dhe kodim one-hot për `c_resid`, `nace_r2` dhe `season`.

In [ ]:
klasat = ["Ulët", "Mesëm", "Lartë", "Shumë i lartë"]
df_features["NR_klasa_frekuence"] = pd.qcut(df_features["NR"], q=4, labels=klasat)
df_features["NR_klasa_gjeresi"] = pd.cut(df_features["NR"], bins=4, labels=klasat)
df_features["NR_log_klasa_gjeresi"] = pd.cut(df_features["NR_log"], bins=4, labels=klasat)

krahasimi_diskretizimit = pd.DataFrame({
    "frekuence_e_barabarte (qcut)": df_features["NR_klasa_frekuence"].value_counts().reindex(klasat),
    "gjeresi_e_barabarte (cut)": df_features["NR_klasa_gjeresi"].value_counts().reindex(klasat),
    "gjeresi_e_barabarte mbi log": df_features["NR_log_klasa_gjeresi"].value_counts().reindex(klasat),
})
display(krahasimi_diskretizimit)
print("Kufijtë e klasave me qcut:", pd.qcut(df_features["NR"], q=4, retbins=True)[1].round(0))

# Binarizimi
df_features["growth_positive"] = np.where(df_features["PCH_SM"].isna(), np.nan, (df_features["PCH_SM"] > 0).astype(float))
df_features["has_flag"] = df_features["ka_flag"].astype(int)
df_features["anetare_e_BE"] = df_features["anetare_e_BE"].astype(int)

dummies = pd.get_dummies(df_features[["c_resid", "nace_r2", "season"]], prefix=["resid", "nace", "stina"], dtype=int)
df_features = pd.concat([df_features, dummies], axis=1)

print("Shpërndarja e growth_positive:")
display(df_features["growth_positive"].value_counts(dropna=False))
print("Kolonat one-hot:", dummies.columns.tolist())
krahasimi_diskretizimit.to_csv(OUTPUT_DIR / "11_discretization_comparison.csv")

### 10.1 Smoothing me binning (trajtimi i zhurmës)

Si në shembullin e ligjëratës: vlerat renditen, ndahen në bin-a me **frekuencë të barabartë** dhe zëvendësohen me **mesataren e bin-it** ose me **kufirin më të afërt të bin-it**. Rreshti mbetet në dataset; ndryshon vetëm vlera ("smoothing ≠ fshirje"). Shembulli: netët mujore në Shqipëri gjatë vitit 2019 (12 vlera, 3 bin-a).

In [ ]:
def smoothing_binning(vlerat, nr_binave=3):
    vlerat = np.sort(np.asarray(vlerat, dtype=float))
    binat = np.array_split(vlerat, nr_binave)
    sipas_mesatares = [np.full(len(b), b.mean()) for b in binat]
    sipas_kufijve = [np.where(b - b.min() <= b.max() - b, b.min(), b.max()) for b in binat]
    return binat, sipas_mesatares, sipas_kufijve


# Verifikim me shembullin e ligjëratës (çmimet në euro)
_, mes_l, kuf_l = smoothing_binning([24, 9, 28, 4, 21, 34, 15, 26, 8, 29, 25, 21])
print("Shembulli i ligjëratës – sipas mesatares:", [[int(round(v)) for v in b] for b in mes_l])
print("Shembulli i ligjëratës – sipas kufijve:  ", [[int(v) for v in b] for b in kuf_l])

vlerat_al = matrica_imputuar.loc["AL", [c for c in matrica_imputuar.columns if c.year == 2019]].values
binat, mes_al, kuf_al = smoothing_binning(vlerat_al)
tabela_binning = pd.DataFrame({
    "bin": np.repeat(["Bin 1", "Bin 2", "Bin 3"], [len(b) for b in binat]),
    "vlera_renditur": np.concatenate(binat),
    "sipas_mesatares": np.concatenate(mes_al).round(0),
    "sipas_kufijve": np.concatenate(kuf_al),
}).astype({"vlera_renditur": int, "sipas_mesatares": int, "sipas_kufijve": int})
display(tabela_binning)

### 10.2 Hierarkitë e koncepteve (roll-up dhe drill-down)

Hierarkitë e koncepteve lejojnë shikimin e të dhënave në nivele të ndryshme të detajit:
- **koha:** muaj < tremujor < vit;
- **hapësira:** shtet < rajon < Evropë – e specifikuar me grupim eksplicit të vlerave, p.sh. {AL, XK, MK, ME, RS} < Ballkani Perëndimor;
- **aktiviteti:** I552, I553 < I552_I553 < I551-I553.

Sipas ligjëratës, hierarkia mund të gjenerohet edhe automatikisht: atributi me më shumë vlera unike vendoset në nivelin më të ulët.

In [ ]:
seria_hierarki = df_features[(df_features["c_resid"] == "TOTAL") & (df_features["nace_r2"] == "I551-I553")].copy()

# Gjenerimi automatik: renditja sipas numrit të vlerave unike
niveli_kohor = pd.Series({
    "period_date (muaji)": seria_hierarki["period_date"].nunique(),
    "quarter (tremujori)": seria_hierarki["period_date"].dt.to_period("Q").nunique(),
    "year (viti)": seria_hierarki["year"].nunique(),
}).sort_values(ascending=False)
niveli_hapesinor = pd.Series({
    "geo (shteti)": seria_hierarki["geo"].nunique(),
    "rajoni": seria_hierarki["rajoni"].nunique(),
}).sort_values(ascending=False)
print("Hierarkia kohore (nga niveli më i ulët):", " < ".join(niveli_kohor.index))
print("Hierarkia hapësinore (nga niveli më i ulët):", " < ".join(niveli_hapesinor.index))

# Roll-up: shtet × muaj → rajon × vit (vetëm vitet e plota 2018–2025)
seria_plote = seria_hierarki[seria_hierarki["year"].between(viti_fillim, viti_fundit)]
roll_up = seria_plote.pivot_table(index="rajoni", columns="year", values="NR", aggfunc="sum") / 1e6
print("\nRoll-up: netët (milionë) sipas rajonit dhe vitit")
display(roll_up.round(1))
print("Rreshta para roll-up:", len(seria_plote), "| qeliza pas roll-up:", roll_up.size)

# Drill-down: Ballkani Perëndimor → shtetet
drill_down = (seria_plote[seria_plote["rajoni"] == "Ballkani Perëndimor"]
              .pivot_table(index="geo", columns="year", values="NR", aggfunc="sum") / 1e6)
print("Drill-down: Ballkani Perëndimor sipas shtetit (milionë netë)")
display(drill_down.round(2))
roll_up.to_csv(OUTPUT_DIR / "11b_concept_hierarchy_rollup.csv")

## 11. Zgjedhja e nënbashkësisë së veçorive

1. Largojmë kolonat **konstante ose pa informacion** (`DATAFLOW`, `LAST UPDATE`, `freq`, `CONF_STATUS` – pas largimit të rreshtave pa vlerë të gjitha vlerat janë publike). Ato u identifikuan në seksionin 4.
2. Largojmë kolonat **e tepërta**: `TIME_PERIOD` (e njëjta informatë si `period_date`), `ka_flag` (zëvendësohet nga `has_flag`), `NR_viti_kaluar` (kolonë ndihmëse).
3. Largojmë veçoritë me **më shumë se 70% mungesa**.
4. Me `VarianceThreshold` largojmë veçoritë numerike me variancë shumë të vogël. Para llogaritjes vlerat kufizohen në përqindjet 1%–99% dhe normalizohen me min-max, që vlerat ekstreme të mos e zvogëlojnë artificialisht variancën.
5. Nga çiftet e veçorive me korrelacion |r| > 0.9 mbajmë vetëm njërën.

In [ ]:
print("Kolonat konstante në datasetin origjinal:", constant_columns)

kolona_te_teperta = ["TIME_PERIOD", "ka_flag", "NR_viti_kaluar"]
df_selected = df_features.drop(columns=kolona_te_teperta, errors="ignore")

numerike = [c for c in df_selected.select_dtypes(include="number").columns
            if c not in ["year", "month", "quarter"]]

# Veçoritë me shumë mungesa
mungesa_vecorive = (df_selected[numerike].isna().mean() * 100).round(2)
me_shume_mungesa = mungesa_vecorive[mungesa_vecorive > 70].index.tolist()
print("Mungesat sipas veçorisë (%):")
display(mungesa_vecorive[mungesa_vecorive > 0].sort_values(ascending=False).to_frame("mungesa_%"))
print("Veçori me më shumë se 70% mungesa:", me_shume_mungesa)
numerike = [c for c in numerike if c not in me_shume_mungesa]

# VarianceThreshold mbi vlerat e kufizuara (1%–99%) dhe të normalizuara
X_num = df_selected[numerike]
X_kufizuar = X_num.clip(lower=X_num.quantile(0.01), upper=X_num.quantile(0.99), axis=1)
X_num_mm = pd.DataFrame(MinMaxScaler().fit_transform(X_kufizuar.fillna(X_kufizuar.median())), columns=numerike)

selektori = VarianceThreshold(threshold=0.01)
selektori.fit(X_num_mm)
varianca = pd.Series(selektori.variances_, index=numerike)
me_variance_te_ulet = varianca[~selektori.get_support()].index.tolist()
print("Varianca e veçorive (pas kufizimit dhe min-max):")
display(varianca.sort_values().round(4).to_frame("varianca"))
print("Veçori me variancë shumë të ulët:", me_variance_te_ulet)

# Korrelacioni
korrelacioni = X_num.drop(columns=me_variance_te_ulet).corr().abs()
trekendeshi = korrelacioni.where(np.triu(np.ones(korrelacioni.shape, dtype=bool), k=1))
cifte_korreluara = (
    trekendeshi.stack().loc[lambda s: s > 0.9].sort_values(ascending=False)
    .rename("abs_r").reset_index().rename(columns={"level_0": "veçoria_1", "level_1": "veçoria_2"})
)
print("Çiftet me |r| > 0.9:")
display(cifte_korreluara)

# Mbajmë veçorinë më të interpretueshme nga çdo çift
prioriteti = ["NR", "PCH_SM", "foreign_share", "is_peak_season", "has_flag", "growth_positive", "anetare_e_BE"]
per_largim = set()
for _, rr in cifte_korreluara.iterrows():
    a, b = rr["veçoria_1"], rr["veçoria_2"]
    if a in per_largim or b in per_largim:
        continue
    mbaje = a if (a in prioriteti and b not in prioriteti) else b if b in prioriteti else a
    per_largim.add(b if mbaje == a else a)

per_largim = sorted(per_largim | set(me_variance_te_ulet) | set(me_shume_mungesa))
print("Veçoritë e larguara:", per_largim)
df_selected = df_selected.drop(columns=per_largim)
print("Forma para zgjedhjes:", df_features.shape, "| pas zgjedhjes:", df_selected.shape)

fig, ax = plt.subplots(figsize=(8, 6.5))
im = ax.imshow(korrelacioni, cmap="viridis", vmin=0, vmax=1)
ax.set_xticks(range(len(korrelacioni)))
ax.set_xticklabels(korrelacioni.columns, rotation=90, fontsize=7)
ax.set_yticks(range(len(korrelacioni)))
ax.set_yticklabels(korrelacioni.columns, fontsize=7)
plt.colorbar(im, ax=ax, label="|r|")
ax.set_title("Matrica e korrelacionit (vlera absolute)")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig_03_korrelacioni.png", dpi=120)
plt.show()
cifte_korreluara.to_csv(OUTPUT_DIR / "12_correlated_feature_pairs.csv", index=False)

### 11.1 Kërkimi heuristik i atributeve: forward selection, backward elimination dhe pema e vendimit

Me d atribute ka 2^d kombinime të mundshme, shumë për t'i provuar të gjitha. Ligjërata paraqet tri metoda heuristike, të cilat kërkojnë një **detyrë** (variabël objektiv). Detyra jonë: të parashikohet **klasa e vëllimit të netëve** (`NR_klasa_frekuence`: Ulët / Mesëm / Lartë / Shumë i lartë, nga seksioni 10) nga atributet që nuk derivohen nga NR. Klasifikuesi është pema e vendimit dhe vlerësimi bëhet me cross-validation 3-fish mbi një mostër të shtresuar prej 20 000 rreshtash.

In [ ]:
dummies_rajoni = pd.get_dummies(df_features["rajoni"], prefix="rajoni", dtype=int)
df_heuristik = pd.concat([df_features, dummies_rajoni], axis=1)

kandidatet = (["foreign_share", "anetare_e_BE", "has_flag", "is_peak_season", "month", "year"]
              + [c for c in df_heuristik.columns if c.startswith(("resid_", "nace_I", "rajoni_"))])
te_dhenat = df_heuristik.dropna(subset=kandidatet + ["NR_klasa_frekuence"])
te_dhenat = te_dhenat.groupby("NR_klasa_frekuence", observed=True, group_keys=False).sample(n=5000, random_state=42)
X_h, y_h = te_dhenat[kandidatet], te_dhenat["NR_klasa_frekuence"].astype(str)
pema = DecisionTreeClassifier(max_depth=6, random_state=42)
K = 5

forward = SequentialFeatureSelector(pema, n_features_to_select=K, direction="forward", cv=3).fit(X_h, y_h)
backward = SequentialFeatureSelector(pema, n_features_to_select=K, direction="backward", cv=3).fit(X_h, y_h)
pema_plote = DecisionTreeClassifier(max_depth=6, random_state=42).fit(X_h, y_h)
rendesia = pd.Series(pema_plote.feature_importances_, index=kandidatet).sort_values(ascending=False)

zgjedhja_forward = list(np.array(kandidatet)[forward.get_support()])
zgjedhja_backward = list(np.array(kandidatet)[backward.get_support()])
zgjedhja_pema = rendesia.head(K).index.tolist()

saktesia = lambda kol: cross_val_score(pema, X_h[kol], y_h, cv=3).mean().round(3)
rezultati_heuristik = pd.DataFrame([
    ("Të gjitha atributet", len(kandidatet), "–", saktesia(kandidatet)),
    ("Forward selection", K, ", ".join(zgjedhja_forward), saktesia(zgjedhja_forward)),
    ("Backward elimination", K, ", ".join(zgjedhja_backward), saktesia(zgjedhja_backward)),
    ("Pema e vendimit (rëndësia)", K, ", ".join(zgjedhja_pema), saktesia(zgjedhja_pema)),
], columns=["metoda", "nr_atributeve", "atributet e zgjedhura", "saktësia_CV"])
display(rezultati_heuristik)
print("Atribute që nuk përdoren fare nga pema (konsiderohen të parëndësishme):",
      rendesia[rendesia == 0].index.tolist())
rezultati_heuristik.to_csv(OUTPUT_DIR / "12b_heuristic_feature_selection.csv", index=False)

Me vetëm 5 atribute saktësia (~51%) është praktikisht e njëjtë me atë që arrihet me të gjitha 20 atributet (~52%). Për krahasim, hamendësimi i rastësishëm me 4 klasa të balancuara jep 25%. Pjesa tjetër e atributeve është pra **e parëndësishme ose redundante** për këtë detyrë. Kjo zgjedhje varet nga detyra, prandaj nuk i largojmë atributet nga dataseti final: ai mbetet i përgjithshëm për fazat e ardhshme.

## 12. Reduktimi i dimensionit (PCA)

Përdorim matricën shtet × muaj të seksionit 4.1 (netët totale pas imputimit). Çdo shtet përshkruhet nga 96 muaj, pra 96 dimensione. PCA zbatohet në dy variante:
1. **PCA mbi vlerat** – `log1p` + standardizim. Gjejmë sa komponentë shpjegojnë 90% të variancës.
2. **PCA mbi profilin** – nga seria logaritmike e çdo shteti zbresim mesataren e tij, që të largohet efekti i madhësisë. Kështu PCA kap formën e serisë: sezonalitetin dhe rënien gjatë pandemisë.

In [ ]:
X_pca = np.log1p(matrica_imputuar)
print("Dimensionet fillestare:", X_pca.shape[1], "muaj për", X_pca.shape[0], "shtete")

# Varianti 1: PCA mbi vlerat
pca = PCA()
pca.fit(StandardScaler().fit_transform(X_pca))
varianca_kumulative = np.cumsum(pca.explained_variance_ratio_)
k_90 = int(np.argmax(varianca_kumulative >= 0.90) + 1)
print("\nVarianti 1 – PCA mbi vlerat")
print("Varianca e shpjeguar nga PC1, PC2, PC3:", (pca.explained_variance_ratio_[:3] * 100).round(2), "%")
print(f"Komponentë të nevojshëm për 90% të variancës: {k_90}")
madhesia_mesatare = X_pca.mean(axis=1)
pc1_vlerat = pca.transform(StandardScaler().fit_transform(X_pca))[:, 0]
print("Korrelacioni PC1 me madhësinë mesatare log(NR):", round(np.corrcoef(pc1_vlerat, madhesia_mesatare)[0, 1], 3))

# Varianti 2: PCA mbi profilin (pa efektin e madhësisë)
X_profil = X_pca.sub(X_pca.mean(axis=1), axis=0)
X_profil_std = StandardScaler().fit_transform(X_profil)
pca_profil = PCA()
komponentet = pca_profil.fit_transform(X_profil_std)
varianca_kum_profil = np.cumsum(pca_profil.explained_variance_ratio_)
k_90_profil = int(np.argmax(varianca_kum_profil >= 0.90) + 1)
print("\nVarianti 2 – PCA mbi profilin")
print("Varianca e shpjeguar nga PC1, PC2, PC3:", (pca_profil.explained_variance_ratio_[:3] * 100).round(2), "%")
print(f"Komponentë të nevojshëm për 90% të variancës: {k_90_profil} (nga {X_pca.shape[1]} dimensione)")

k_ruaj = max(k_90_profil, 2)
pca_rezultati = pd.DataFrame(komponentet[:, :k_ruaj], index=X_pca.index,
                             columns=[f"PC{i + 1}" for i in range(k_ruaj)])
pca_rezultati = pca_rezultati.join(metadata_shtetet.set_index("geo")[["emri_shtetit", "rajoni"]])

# Amplituda sezonale (korrik/janar) për të interpretuar PC1
amplituda = (X_pca.loc[:, [c for c in X_pca.columns if c.month in (7, 8)]].mean(axis=1)
             - X_pca.loc[:, [c for c in X_pca.columns if c.month in (1, 2)]].mean(axis=1))
print("Korrelacioni PC1 (profil) me amplitudën sezonale verë–dimër:",
      round(np.corrcoef(pca_rezultati["PC1"], amplituda)[0, 1], 3))

# Rënia gjatë pandemisë (mesatarja log 2020 – mesatarja log 2019) për të interpretuar PC2
mesatarja_vitit = lambda viti: X_pca.loc[:, [c for c in X_pca.columns if c.year == viti]].mean(axis=1)
renia_covid = mesatarja_vitit(2020) - mesatarja_vitit(2019)
print("Korrelacioni PC2 (profil) me ndryshimin 2019→2020:",
      round(np.corrcoef(pca_rezultati["PC2"], renia_covid)[0, 1], 3))
display(pca_rezultati[["emri_shtetit", "rajoni", "PC1", "PC2"]].sort_values("PC1").round(3))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
nr_komp = 15
for ax, model, titulli in [(axes[0], pca, "Varianti 1: vlerat"), (axes[1], pca_profil, "Varianti 2: profili")]:
    vk = np.cumsum(model.explained_variance_ratio_)
    ax.bar(range(1, nr_komp + 1), model.explained_variance_ratio_[:nr_komp] * 100, color="tab:blue", label="individuale")
    ax.plot(range(1, nr_komp + 1), vk[:nr_komp] * 100, "o-", color="tab:orange", label="kumulative")
    ax.axhline(90, linestyle="--", color="gray")
    ax.set_xlabel("Komponenti")
    ax.set_ylabel("Varianca e shpjeguar (%)")
    ax.set_title(f"Varianca e shpjeguar – {titulli}")
    ax.legend()

for rajoni, grup in pca_rezultati.groupby("rajoni"):
    axes[2].scatter(grup["PC1"], grup["PC2"], label=rajoni, s=40)
for geo, rr in pca_rezultati.iterrows():
    axes[2].annotate(geo, (rr["PC1"], rr["PC2"]), fontsize=7, xytext=(3, 3), textcoords="offset points")
axes[2].set_xlabel(f"PC1 ({pca_profil.explained_variance_ratio_[0] * 100:.1f}%)")
axes[2].set_ylabel(f"PC2 ({pca_profil.explained_variance_ratio_[1] * 100:.1f}%)")
axes[2].set_title("Shtetet në hapësirën PC1–PC2 (profili)")
axes[2].legend(fontsize=7)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig_04_pca.png", dpi=120)
plt.show()

pca_rezultati.to_csv(OUTPUT_DIR / "13_pca_components.csv")

**Interpretimi:**
- **Varianti 1:** PC1 shpjegon rreth 94% të variancës dhe korrelacioni i tij me madhësinë mesatare të shtetit është ≈ 1. PCA mbi vlerat e papërpunuara kap pra kryesisht madhësinë e sektorit turistik, jo dallimet në sjellje.
- **Varianti 2:** pas heqjes së efektit të madhësisë, 96 dimensionet reduktohen në 7 komponentë që ruajnë 90% të informacionit.
  - **PC1 (~50%)** përputhet me amplitudën sezonale verë–dimër (r ≈ 0.98). Vlerat e larta i kanë shtetet me turizëm të përqendruar në verë (HR, EL, ME, CY, AL). Vlerat e ulëta i kanë shtetet me turizëm të shpërndarë gjatë vitit ose me turizëm dimëror (FI, AT, CH, XK).
  - **PC2 (~16%)** lidhet me rënien gjatë pandemisë 2019→2020 (r ≈ −0.73). Shtetet ishullore dhe ato që varen nga fluturimet (CY, MT, IE) kanë PC2 të lartë, pra rënie më të thellë.

### 12.1 Kompresimi me wavelet (Haar) dhe përmbledhja e reduktimit të të dhënave

Transformimi diskret wavelet (DWT) zëvendëson çdo çift vlerash me **mesataren** dhe **gjysmën e diferencës**. Kështu fitohen koeficientë të së njëjtës gjatësi. Nëse mbajmë vetëm koeficientët më të fortë dhe të tjerët i bëjmë 0, fitojmë një përafrim të kompresuar të serisë (**reduktim me humbje**). Implementimin e verifikojmë fillimisht me shembullin e ligjëratës.

In [ ]:
def haar_dwt(x):
    x = np.asarray(x, dtype=float)
    detajet = []
    while len(x) > 1:
        mesataret = (x[0::2] + x[1::2]) / 2
        detajet = list((x[0::2] - x[1::2]) / 2) + detajet
        x = mesataret
    return np.array(list(x) + detajet)


def haar_idwt(koef):
    koef = np.asarray(koef, dtype=float)
    x, poz = koef[:1], 1
    while poz < len(koef):
        d = koef[poz:2 * poz]
        x = np.column_stack([x + d, x - d]).ravel()
        poz *= 2
    return x


print("Shembulli i ligjëratës:", haar_dwt([2, 2, 0, 2, 3, 5, 4, 4]))
print("Rindërtimi:", haar_idwt(haar_dwt([2, 2, 0, 2, 3, 5, 4, 4])))

# Seria e Shqipërisë: 64 muajt e fundit (64 = 2^6), në shkallë logaritmike
seria_al = np.log1p(matrica_imputuar.loc["AL"].iloc[-64:])
koef_al = haar_dwt(seria_al.values)

rezultatet_wavelet = []
for k in [4, 8, 16, 32, 64]:
    koef_k = koef_al.copy()
    koef_k[np.argsort(np.abs(koef_k))[:-k]] = 0          # mbajmë k koeficientët më të fortë
    rindertimi = np.expm1(haar_idwt(koef_k))
    gabimi = np.mean(np.abs(rindertimi - np.expm1(seria_al.values)) / np.expm1(seria_al.values)) * 100
    rezultatet_wavelet.append((k, f"{64 / k:.0f}:1", round(gabimi, 2)))
    if k == 16:
        rindertimi_16 = rindertimi
display(pd.DataFrame(rezultatet_wavelet, columns=["koeficientë të mbajtur", "raporti i kompresimit", "gabimi_mesatar_%"]))

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(seria_al.index, np.expm1(seria_al.values), label="origjinale (64 vlera)")
ax.plot(seria_al.index, rindertimi_16, "--", label="wavelet Haar (16 koeficientë)")
ax.set_title("Shqipëria – kompresimi i serisë me wavelet Haar")
ax.set_ylabel("Netë qëndrimi")
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig_05_wavelet.png", dpi=120)
plt.show()

**Strategjitë e reduktimit të të dhënave të përdorura në projekt (java V):**

| Strategjia | Ideja | Ku u përdor |
|---|---|---|
| Reduktimi i dimensionit | Më pak atribute | PCA (12), zgjedhja e veçorive (11, 11.1) |
| Reduktimi i numrit (numerosity) | Më pak objekte ose një model i tyre | Mostrimi (5), agregimi vjetor dhe data cube / roll-up (6, 10.2), histogramet / binning (10) |
| Kompresimi | Më pak bit | Wavelet Haar (12.1) – me humbje |

## 13. Përmbledhja dhe eksporti final

In [ ]:
permbledhja = pd.DataFrame([
    ("Mbledhja e të dhënave", "1, 1.1", "II", "CSV + API e Eurostat, licenca, kontrolle me regex"),
    ("Definimi i tipeve të dhënave", "2, 2.1", "III", "Tipet në pandas, nominal/ordinal/interval/raport, lloji i datasetit"),
    ("Kualiteti i të dhënave", "3, 3.1", "III, IV", "Gjashtë dimensionet e cilësisë, konsistenca e totaleve, mungesat, dublikatat, IQR"),
    ("Pastrimi", "4", "IV", "Heqja e hapësirave, konvertimi numerik dhe i datës, kolonat konstante"),
    ("Vlerat e zbrazëta", "4.1", "III, IV", "Analiza sipas statusit, krahasimi i 4 metodave, interpolim + mesatarja e klasës"),
    ("Vlerat ekstreme / anomalitë", "4.2", "III, VI", "Ruajtja e tyre, winsorizimi, |z| > 3 kontekstual"),
    ("Mostrimi", "5, 5.1", "III, V", "Mostër e rastësishme dhe e shtresuar"),
    ("Agregimi", "6, 10.2", "III, V", "Agregimi vjetor, stabiliteti (CV), roll-up / drill-down"),
    ("Integrimi", "7, 7.1", "IV", "Metadata e shteteve, identifikimi i entiteteve, χ², korrelacioni, kovarianca"),
    ("Krijimi i veçorive", "8", "V", "Veçori kohore, foreign_share, yoy_change_calc, rolling_3m_mean"),
    ("Transformimi", "9", "V", "log1p, z-score, min-max, decimal scaling"),
    ("Diskretizimi dhe binarizimi", "10, 10.1, 10.2", "V", "Gjerësi/frekuencë e barabartë, smoothing me binning, hierarki, 0/1, one-hot"),
    ("Zgjedhja e veçorive", "11, 11.1", "III, V", "Filtra (mungesat, varianca, korrelacioni), forward, backward, pema e vendimit"),
    ("Reduktimi i dimensionit", "12, 12.1", "III, V", "PCA (dy variante), wavelet Haar"),
], columns=["Pika e detyrës", "Seksioni", "Java e ligjëratës", "Çfarë u bë"])
display(permbledhja)

df_selected.to_csv(OUTPUT_DIR / "final_preprocessed_dataset.csv", index=False)
print("Dataseti final:", df_selected.shape)
print("Skedarët e eksportuar në", OUTPUT_DIR, ":")
for f in sorted(OUTPUT_DIR.iterdir()):
    print("  -", f.name)